# Week 3, day 4 (morning) — Worksheet 02 SOLUTIONS: grain

Executed in the lab image against the generated source data. Every quoted number
is what it actually printed.

Questions 2 and 3 are the ones that matter. The grain the lecture states is not
unique in the source, and what you do about that is a modelling decision with a
visible consequence either way.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 02 — Grain. Run this once.
import pandas as pd

DATA = "data/"
load = lambda name: pd.read_csv(DATA + name + ".csv")

enr = load("enrollment")
tx = load("transaction")

print("enrollment: ", enr.shape)
print("transaction:", tx.shape)
print()
print("the grain slide 27 states:")
print("  one row per student's enrollment in a specific course and cohort")

PART A — is the stated grain real?

### Question 1

A grain is a uniqueness claim. Test several candidate grains on `enrollment`: for each of `[stu_id, course_id, cohort_id]`, `[stu_id, course_id]`, `[course_id, cohort_id]` and `[enrl_id]`, print the number of distinct combinations and how many rows are duplicates.

In [ ]:
candidates = [
    ["stu_id", "course_id", "cohort_id"],
    ["stu_id", "course_id"],
    ["course_id", "cohort_id"],
    ["enrl_id"],
]
print("rows:", len(enr))
print()
for cols in candidates:
    n = len(enr.drop_duplicates(cols))
    print("  %-38s %5d distinct, %5d duplicated"
          % (" + ".join(cols), n, len(enr) - n))

```
rows: 2400

  stu_id + course_id + cohort_id          2384 distinct,    16 duplicated
  stu_id + course_id                      2233 distinct,   167 duplicated
  course_id + cohort_id                    383 distinct,  2017 duplicated
  enrl_id                                 2400 distinct,     0 duplicated
```

Only `enrl_id` is unique. **The grain the lecture states is not** — 16 rows
repeat a `student + course + cohort` combination that appeared earlier.

Sixteen rows out of 2,400 -- well under one percent -- which is exactly the size
of problem that never gets noticed. It is too small to distort a chart, too small to trip a
reconciliation, and large enough to break any process that assumes the key is a
key: a `MERGE ... ON (stu_id, course_id, cohort_id)`, a `pivot`, a
`drop_duplicates` intended to be a no-op.

Note how cheap this test is. Three lines, no domain knowledge, and it can be run
against any table the moment you receive it. **A grain statement is a uniqueness
claim, and a uniqueness claim is checkable.** The lecture states the grain and
moves on; a modelling decision you have not tested is a guess with a sentence
around it.

Questions 2 and 3 work out whether the 16 are a defect or a misunderstanding.

### Question 2

The lecture's grain is `student + course + cohort`. That combination is **not** unique. Count the rows involved, and print the offending rows in full, sorted so the pairs sit together.
> **NOTE:** before calling them duplicates, look at `enrl_id` and `enrl_date`.

In [ ]:
key = ["stu_id", "course_id", "cohort_id"]
dupes = enr[enr.duplicated(key, keep=False)].sort_values(key + ["enrl_date"])
print("rows breaking the stated grain:", len(dupes))
print("distinct student/course/cohort combinations involved:",
      len(dupes.drop_duplicates(key)))
print()
print(dupes.head(10).to_string(index=False))

```
rows breaking the stated grain: 32
distinct student/course/cohort combinations involved: 16
```

Two numbers for the same finding, and the difference is worth being precise
about. `duplicated(keep=False)` marks **both** members of each colliding pair, so
32 rows are involved. `duplicated()` on its own marks only the second occurrence,
which is the 16 from question 1. Sixteen collisions, thirty-two rows.

The rows themselves:

```
 enrl_id  enrl_date  stu_id  course_id  cohort_id status
  701580 2024-02-24    5034        214        303 active
  701146 2024-03-07    5034        214        303 active
  701070 2025-04-11    5130        210        313 active
  700345 2025-04-20    5130        210        313 active
```

And now the thing that decides everything: **the two rows in each pair have
different `enrl_id` values and different `enrl_date` values.**

That rules out the easy explanation. A duplicate load produces identical rows —
same id, same date, same everything — and is a data-quality defect you delete.
These are not identical. They were created at different times, by the operational
system, as separate records.

So the source is not broken. The *grain sentence* is imprecise. Question 3
establishes what actually happened.

### Question 3

Decide what they are. For those rows print, per pair: the two `enrl_id` values, the two `enrl_date` values, the gap in days between them, and the two `status` values.
> **NOTE:** a duplicate load produces identical rows. Two separate business events do not.

In [ ]:
key = ["stu_id", "course_id", "cohort_id"]
dupes = enr[enr.duplicated(key, keep=False)].copy()
dupes["enrl_date"] = pd.to_datetime(dupes["enrl_date"])
for (s, c, co), g in dupes.groupby(key):
    g = g.sort_values("enrl_date")
    gap = (g["enrl_date"].iloc[1] - g["enrl_date"].iloc[0]).days
    print("stu %d course %d cohort %d | ids %s | dates %s | %3d days apart | %s"
          % (s, c, co, list(g.enrl_id),
             [d.date().isoformat() for d in g.enrl_date], gap,
             list(g.status)))
print()
print("identical rows ignoring enrl_id:",
      int(dupes.drop(columns=["enrl_id"]).duplicated().sum()))

Every pair is separated by days or weeks, and two of them change status:

```
stu 5034 course 214 cohort 303 | ids [701580, 701146] | dates ['2024-02-24', '2024-03-07'] |  12 days apart | ['active', 'active']
stu 5307 course 202 cohort 302 | ids [700442, 701641] | dates ['2024-02-02', '2024-02-04'] |   2 days apart | ['active', 'active']
stu 5434 course 206 cohort 301 | ids [701407, 700417] | dates ['2023-12-29', '2024-01-06'] |   8 days apart | ['cancelled', 'active']
stu 5458 course 224 cohort 312 | ids [702295, 700851] | dates ['2025-03-23', '2025-03-25'] |   2 days apart | ['cancelled', 'active']
stu 5583 course 218 cohort 302 | ids [701209, 701994] | dates ['2024-01-10', '2024-02-11'] |  32 days apart | ['active', 'active']
```

and:

```
identical rows ignoring enrl_id: 0
```

**Not one pair is identical.** These are real, separate business events.

Look at student 5434 and student 5458. In each case the first enrollment is
**cancelled** and the second is **active** — a student who cancelled and then
signed up again for the same course and cohort. That is not a data problem; that
is a customer changing their mind, and it is exactly the kind of thing the
business will want to count.

The other fourteen pairs are two active enrollments in the same course and
cohort, days apart. Depending on the operational system that is a duplicate
registration the admissions team will resolve, a re-registration after a
withdrawal that was never flagged, or an administrative correction. **You cannot
tell from the data, and you should not guess.** That is a question for whoever
owns the source system.

What you can decide is what the model does about it, and there are only two
honest options:

**Keep both rows.** The grain becomes *one row per enrollment event*, `enrl_id`
is the key, and re-enrollments are counted twice — which is correct if the
question is "how many enrollment events did we process".

**Collapse each pair to one.** The grain stays as slide 27 states it, and you
must then write down which of the two survives (the latest? the active one?) and
what happens to the other one's payments.

Either is defensible. What is not defensible is leaving the sentence as it is
while the data says otherwise, because every downstream engineer will read the
sentence and trust it.

Question 4 takes the first option, and says so.

### Question 4

Restate the grain so it is true. Confirm `enrl_id` is unique and not null, and write the corrected one-sentence grain.

In [ ]:
print("rows:              ", len(enr))
print("distinct enrl_id:  ", enr.enrl_id.nunique())
print("null enrl_id:      ", int(enr.enrl_id.isna().sum()))
print("unique:            ", enr.enrl_id.is_unique)
print()
print("corrected grain:")
print("  one row per enrollment EVENT -- identified by enrl_id --")
print("  of one student in one course and one cohort")

```
rows:               2400
distinct enrl_id:   2400
null enrl_id:       0
unique:             True
```

`enrl_id` is unique, complete, and therefore usable as the fact table's primary
key.

The corrected sentence:

> **One row represents one enrollment event — identified by `enrl_id` — of one
> student in one course and one cohort.**

The difference from slide 27 is four words, and it is the difference between a
claim the data supports and one it does not. "One student's enrollment in a
specific course and cohort" implies that combination identifies the row.
"One enrollment event, identified by `enrl_id`" says what is actually true, and
leaves room for the re-enrollments found in question 3.

Two consequences follow immediately, and both are the point of stating a grain at
all:

**`enrollment_count` is now well defined.** It is 1 per row, and summing it
counts enrollment *events*. If the business later asks for distinct students on a
course, that is `COUNT(DISTINCT student_id)` and a different question — but at
least it is now visibly a different question rather than an ambiguity.

**The fact table has 2,400 rows**, not 2,384 and not 2,243. Worksheet 09 loads it
and worksheet 10 checks that number, and both of those checks are only possible
because the grain was pinned down here.

Write the grain down where the next engineer will find it — in the table comment,
in the model documentation, in the ELT spec. Slide 41 lists "target grain and
aggregation logic" as something an ETL specification must document, and this is
why: the grain is not recoverable from the code, and getting it wrong is not
visible in the output.

PART B — what a different grain would cost

### Question 5

Try a **coarser** grain: aggregate enrollments to one row per `course_id + cohort_id`, counting enrollments. Print the row count, and the top five rows.

In [ ]:
coarse = (enr.groupby(["course_id", "cohort_id"]).size()
             .rename("enrollment_count").reset_index())
print("rows at course+cohort grain:", len(coarse))
print("rows at enrollment grain:   ", len(enr))
print("compression: %.1fx" % (len(enr) / len(coarse)))
print()
print(coarse.sort_values("enrollment_count", ascending=False)
            .head(5).to_string(index=False))

```
rows at course+cohort grain: 383
rows at enrollment grain:    2400
compression: 6.3x

 course_id  cohort_id  enrollment_count
       223        309                15
       223        314                14
       224        303                14
       206        306                13
       204        316                13
```

383 rows instead of 2,400 — a table 6.3 times smaller that answers "which
course-cohort groups are biggest" perfectly well.

This is a real and often correct design. Slide 10 lists *"one row per daily
store-product sales summary"* as a legitimate fact table grain alongside the
detailed ones. Pre-aggregated fact tables are smaller, faster, and cheaper to
query, and for a dashboard that only ever shows course-cohort totals, this is the
better table.

The cost is not performance. It is that **you can never get the detail back.**
Aggregation is one-way: once 2,400 rows are 383, no query can recover which
student enrolled or on what date, because those columns are gone.

Which is why the standard advice is to build the fact table at **the finest grain
the source supports**, and aggregate from it — not to it. A detailed fact table
can produce the summary; the summary can never produce the detail. Storage is
cheap and re-extracting last year's source data usually is not possible at all.

Question 6 makes the loss concrete against the lecture's own business questions.

### Question 6

Slide 25 lists six business questions. At the coarse grain from question 5, which can still be answered? Check two concretely: enrollments per day, and enrollments per student.
> **NOTE:** the coarse table has no `enrl_date` and no `stu_id`. That is the answer, but show it.

In [ ]:
coarse = (enr.groupby(["course_id", "cohort_id"]).size()
             .rename("enrollment_count").reset_index())
print("columns available at the coarse grain:", list(coarse.columns))
print()
for need, col in [("enrollments per day", "enrl_date"),
                  ("enrollments per student", "stu_id"),
                  ("enrollments per course", "course_id")]:
    print("  %-26s needs %-11s -> %s"
          % (need, col, "yes" if col in coarse.columns else "IMPOSSIBLE"))
print()
print("at enrollment grain, enrollments per day:")
print(enr.groupby("enrl_date").size().sort_values(ascending=False)
         .head(3).rename("enrollments").to_string())
print("distinct enrollment dates:", enr.enrl_date.nunique())

```
columns available at the coarse grain: ['course_id', 'cohort_id', 'enrollment_count']

  enrollments per day        needs enrl_date   -> IMPOSSIBLE
  enrollments per student    needs stu_id      -> IMPOSSIBLE
  enrollments per course     needs course_id   -> yes
```

Two of the three are unanswerable, and the first one — *"How many students
enrolled each day?"* — is **question one on slide 25**. The coarse grain fails the
very first requirement the model was built for.

At enrollment grain it is trivial:

```
enrl_date
2025-07-06    14
2024-08-03    14
2024-12-07    12

distinct enrollment dates: 650
```

650 distinct enrollment dates, peaking at 14 on a day. Those 650 rows of daily
detail are precisely what the aggregation threw away.

This is why slide 22's practical process puts *"identify business questions"*
first and *"validate against the questions"* last, with the grain decision in
between. The grain is chosen to serve the questions, and then checked against
them again at the end. Worksheet 10 runs that final check.

The general rule, and it is a strong one:

> **The grain must be at least as fine as the finest question you need to
> answer.**

And since the questions you will be asked in a year are not knowable now, "as
fine as the source allows" is the safe default. Slide 25's sixth question —
*"which course-cohort groups may be over-discounted or underperforming?"* — did
not exist when the enrollment table was designed either.

### Question 7

Now try a **finer** grain: join `enrollment` to `transaction`, which is one row per payment. Count enrollments from that table two ways — `len()` and `nunique()` on `enrl_id` — and print both against the true figure.

In [ ]:
fine = enr.merge(tx, on="enrl_id")
print("rows at payment grain:      ", len(fine))
print("COUNT(*)          reports:  ", len(fine), "enrollments  <- wrong")
print("COUNT(DISTINCT)   reports:  ", fine.enrl_id.nunique(), "enrollments")
print("true enrollment count:      ", len(enr))
print()
print("payment grain overstates by %.2fx" % (len(fine) / len(enr)))
print("and DISTINCT still misses   ", len(enr) - fine.enrl_id.nunique(),
      "enrollments that have no payment")

```
rows at payment grain:       4856
COUNT(*)          reports:   4856 enrollments  <- wrong
COUNT(DISTINCT)   reports:   2243 enrollments
true enrollment count:       2400

payment grain overstates by 2.02x
and DISTINCT still misses    157 enrollments that have no payment
```

Three different answers to "how many enrollments", from one join.

**4,856** is what `COUNT(*)` gives, and it is nonsense — it counts payments and
calls them enrollments, overstating by 2.02x.

**2,243** is what `COUNT(DISTINCT enrl_id)` gives. Better, and still wrong: it
counts enrollments *that have at least one payment*, silently excluding the 157
that have none.

**2,400** is the truth, and it is only available from the `enrollment` table
itself.

The instructive part is that `COUNT(DISTINCT)` — the fix everyone reaches for —
does not get you there. It repairs the fan-out and leaves the filter, because the
inner join already deleted the rows before `DISTINCT` could see them. Two
different errors in one query, and correcting the obvious one makes the result
look trustworthy while it is still 157 short.

That is the case for a fact table at a declared grain. In `fact_enrollment`,
`COUNT(*)` is the enrollment count, always, because the table has exactly one row
per enrollment by construction. No analyst needs to remember to write `DISTINCT`,
and no analyst needs to know that 157 enrollments never paid.

**The grain is a promise the table makes so that queries do not have to.**

PART C — grain decides which measures are valid

### Question 8

`full_price` and `payment_amount` both live in `transaction`. Sum each at payment grain and at enrollment grain, and print all four totals.
> **NOTE:** one of these columns describes the enrollment; the other describes the payment. Only one is safe to sum where it sits.

In [ ]:
print("at PAYMENT grain (%d rows):" % len(tx))
print("  SUM(full_price)     %15.2f" % tx.full_price.sum())
print("  SUM(payment_amount) %15.2f" % tx.payment_amount.sum())
print()
per_enr = tx.groupby("enrl_id").agg(full_price=("full_price", "max"),
                                    paid=("payment_amount", "sum"))
print("at ENROLLMENT grain (%d rows):" % len(per_enr))
print("  SUM(full_price)     %15.2f" % per_enr.full_price.sum())
print("  SUM(paid)           %15.2f" % per_enr.paid.sum())

```
at PAYMENT grain (4856 rows):
  SUM(full_price)         25073000.00
  SUM(payment_amount)      8953821.53

at ENROLLMENT grain (2243 rows):
  SUM(full_price)         11628000.00
  SUM(paid)                8953821.53
```

Look at the two rows rather than the four numbers.

**`payment_amount` is identical at both grains: 8,953,821.53.** Rolling 4,856
payments up into 2,243 enrollments changed nothing, because each payment is
counted exactly once either way.

**`full_price` is not: 25,073,000.00 against 11,628,000.00**, a factor of 2.16.

The difference is what the column *describes*. `payment_amount` describes the
payment — the row it sits on — so it is genuinely additive at payment grain.
`full_price` describes the **enrollment**, and the operational system merely
repeats it on each of that enrollment's payment rows for convenience. Summing it
at payment grain counts the same tuition once per instalment.

This is the precise meaning of slide 11's **additive** measure — *"can be summed
across all relevant dimensions at the defined grain"* — and the four words that
matter are *at the defined grain*. Additivity is not a property of a column. It
is a property of a column **and** a grain, and a column can be additive at one
and meaningless at another.

The practical test, which costs one line:

> Roll the measure up to the coarser grain and sum it. If the total changes, it
> was not additive where it was.

`payment_amount` passes. `full_price` fails, and the right way to bring it to
enrollment grain is `max()` or `first()`, never `sum()`.

Worksheet 03 takes this further into semi-additive and non-additive measures,
where even the correct grain does not make `SUM` safe.

### Question 9

Show what a mixed-grain table looks like. Build one row per enrollment carrying both `enrollment_count = 1` and the enrollment's `full_price`, then `concat` it with the payment-grain rows and sum both columns. Print the totals and say which is now meaningless.
> **NOTE:** this is the mistake that produces a table nobody can safely aggregate. Make it, then look at it.

In [ ]:
at_enr = tx.groupby("enrl_id").agg(full_price=("full_price", "max")).reset_index()
at_enr["enrollment_count"] = 1
at_pay = tx[["enrl_id", "full_price"]].copy()
at_pay["enrollment_count"] = 1

mixed = pd.concat([at_enr, at_pay], ignore_index=True)
print("enrollment-grain rows:", len(at_enr))
print("payment-grain rows:   ", len(at_pay))
print("mixed table rows:     ", len(mixed))
print()
print("SUM(enrollment_count) %8d   <- should be %d" % (
    mixed.enrollment_count.sum(), tx.enrl_id.nunique()))
print("SUM(full_price)   %15.2f   <- should be %.2f" % (
    mixed.full_price.sum(), at_enr.full_price.sum()))

```
enrollment-grain rows: 2243
payment-grain rows:    4856
mixed table rows:      7099

SUM(enrollment_count)     7099   <- should be 2243
SUM(full_price)       36701000.00   <- should be 11628000.00
```

Every column in this table is now wrong, and the table itself looks completely
normal: three columns, no nulls, no duplicate ids, sensible values in every row.

`SUM(enrollment_count)` reports **7,099** enrollments for a business that had
2,243 of them, because 2,243 enrollment rows and 4,856 payment rows each
contributed a 1. `SUM(full_price)` reports **36.7 million** against a true 11.6
million.

Nothing about the table reveals this. There is no `grain` column. There is no
constraint that could have been violated. Two sets of rows that mean different
things are sitting in one table looking exactly alike, and the only way to know
is to have been told.

This is the concrete version of slide 9's warning — *"if the grain is unclear,
the fact table will be difficult to query and easy to misinterpret"* — and it is
usually created by accident rather than by a `concat` this obvious. The realistic
routes in are a `UNION ALL` that appends a summary to a detail table, an
incremental load that inserts at a different grain from the initial load, or a
"just add the totals as extra rows" request from someone who wants them
convenient.

Two defences, and you want both:

**Never mix grains in one table.** If you need enrollment-level and payment-level
facts, that is `fact_enrollment` and `fact_payment` — two tables sharing
dimensions. Slide 29's own note says exactly this: *"For detailed payment
analysis, add a separate fact_payment table. This would turn the model into a
galaxy schema."* Worksheet 04 builds it.

**Make the grain checkable.** `fact_enrollment` has `enrollment_id` as a unique
primary key, so a row at the wrong grain either collides or is obviously extra.
A fact table with no key is a fact table where this failure is undetectable.

### Question 10

Finally, make pandas enforce the grain for you. `pivot` requires its index/column pair to be unique — run `enr.pivot(index=["stu_id", "cohort_id"], columns="course_id", values="enrl_id")`. **This is supposed to fail.** Connect the error to question 2.

In [ ]:
key = ["stu_id", "course_id", "cohort_id"]
print("rows breaking student+course+cohort:",
      int(enr.duplicated(key, keep=False).sum()))
print()
print(enr.pivot(index=["stu_id", "cohort_id"], columns="course_id",
                values="enrl_id").shape)

```
rows breaking student+course+cohort: 32

ValueError: Index contains duplicate entries, cannot reshape
```

`pivot` needs each index/column pair to identify exactly one value. It found the
16 collisions from question 2 and refused.

That is the whole worksheet in one error message. **`pivot` is the only operation
here that checks the grain claim and stops when it is false** — every other
operation on this data was perfectly happy:

- the three-table join in worksheet 01 succeeded and dropped nothing
- `groupby(["stu_id", "course_id", "cohort_id"]).size()` would have silently
  returned 2,384 rows instead of 2,400
- `merge` on those three columns would have quietly produced extra rows
- `drop_duplicates` on them would have deleted 16 real enrollments

None of those raise. All of them are wrong in the same way, and only this one
says so.

So the practical advice is not "use pivot". It is: **you cannot rely on your
tools to notice a grain error, so test the grain yourself, on arrival, in one
line**:

```python
assert df.duplicated(GRAIN_COLUMNS).sum() == 0, "grain violated"
```

Put it in the load. It is the cheapest assertion in the pipeline and it fails
loudly, which is more than anything else in this worksheet did.

**What this sheet established:**

| | |
|---|---|
| the stated grain | 16 collisions, 32 rows — not unique |
| what they were | real re-enrollments, two after a cancellation — not duplicates |
| the corrected grain | one row per enrollment event, keyed by `enrl_id` |
| too coarse (383 rows) | loses slide 25's first business question entirely |
| too fine (4,856 rows) | three different answers to "how many enrollments" |
| `full_price` at the wrong grain | 25.07M instead of 11.63M |
| grains mixed in one table | every column wrong, nothing detectable |

Worksheet 03 uses the grain to sort the columns into measures and attributes —
and finds that even at the right grain, not every measure can be summed.